# 04 Qualitative examples (inference only)
Retrains the seven models of plan A for fold 0 and seed 0 (Q4, Qp4, Tr4, Cl4, Id4, Full, U-Net) with the same code, initial weights and seed as the main runs, checks the BUSI test Dice against the stored run, and saves the masks of four example images chosen by a fixed rule from the stored results (see `src/qual_run.py`). A Q4 checkpoint from the noise test is reused when it exists.
T4 GPU only. One session is enough (about 1.5 to 3 hours). A second session may run cell 2 with `SESSION = 'B'`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time
ROOT = '/content/drive/MyDrive/BUSI_QB'
CODE = os.path.join(ROOT, 'code'); PREP = os.path.join(ROOT, 'prepared'); RUNS = os.path.join(ROOT, 'runs')
NOISE = os.path.join(ROOT, 'runs_noise'); QUAL = os.path.join(ROOT, 'runs_qual')
sys.path.insert(0, os.path.join(CODE, 'src'))
SESSION = 'A'                                     # 'B' in a second session
import numpy as np, pandas as pd, tensorflow as tf
import qsim, qbus, runner, qual_run
print(qbus.env_info())
assert 'T4' in str(qbus.env_info().get('gpu')), 'select a T4 GPU runtime'
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)
os.makedirs(QUAL, exist_ok=True)
dec = json.load(open(os.path.join(RUNS, 'plan_decision.json')))
K = {m: (1 if qbus.parse(m)[0] in ('id', 'full', 'unet') else dec['selected'][m]) for m in qual_run.MODELS}
SEL_FILE = os.path.join(QUAL, 'selected_examples.json')
if not os.path.exists(SEL_FILE):
    json.dump(qual_run.select_examples(RUNS), open(SEL_FILE, 'w'), indent=1)
SEL = json.load(open(SEL_FILE))
print('k:', K); print(json.dumps({d: v['image'] for d, v in SEL.items()}))

In [ ]:
# 2. RUNS --------------------------------------------------------------------------------------------
todo = list(qual_run.MODELS)
for m in (reversed(todo) if SESSION == 'B' else todo):
    rid = qual_run.rid(m)
    if os.path.exists(os.path.join(QUAL, rid, 'summary.json')):
        continue
    if runner.claim(QUAL, rid, SESSION):
        print(time.strftime('%m-%d %H:%M'), 'start', rid)
        s = qual_run.one_model(m, PREP, RUNS, NOISE, QUAL, SEL, K[m])
        print('   ', s['source'], 'Dice', round(s['busi_dice'], 4), 'stored', round(s['stored_dice'], 4),
              'max image diff', round(s['max_abs_image_diff'], 4), 'min', round(s['minutes'], 1))

In [ ]:
# 3. COLLECT (when all seven are done) ------------------------------------------------------------------
done = [m for m in qual_run.MODELS if os.path.exists(os.path.join(QUAL, qual_run.rid(m), 'summary.json'))]
print(f'done {len(done)} / 7')
if len(done) == 7:
    os.makedirs(os.path.join(ROOT, 'analysis'), exist_ok=True)
    T = qual_run.collect(PREP, QUAL, SEL, os.path.join(ROOT, 'analysis', 'qualitative_examples.npz'))
    T.to_csv(os.path.join(ROOT, 'analysis', 'qualitative_runs.csv'), index=False)
    print(T.round(4).to_string())
    print('QUAL COLLECT OK')